# Agent State Machine (ASM) - Quick Start Guide

An **agent** is a wrapper around a finite state machine designed to accomplish a specific task and will be referred to as ASM.

A **StateMachineBuilder** is used to build the ASM from a manifest and a state diagram.

```mermaid
flowchart TD
    A["state_diagram"] -->|Input| B["StateMachineBuilder"]
    A2["state_manifest"] -->|Input| C["StateModel"]
    C -->|Build| B
    B --> D["fsm"]
```


---

## Example: Standard Assistant

In this example, we will demonstrate how to create a simple **Assistant** agent using a state diagram and state manifest.

### a) Define State Diagram

The following is a simple example of an **Assistant** agent using 3 states:

* **INIT:** Collect initial input.
* **GENERATE:** The state where the LLM generates a response.
* **FINAL:** Return final output.

```mermaid
stateDiagram-v2
direction LR
INIT --> GENERATE: next / action
GENERATE --> FINAL: next / action
```


In [1]:
STATE_DIAGRAM = """
    INIT --> GENERATE
    GENERATE --> FINAL
    """

### b) Define State Manifest

The state manifest is a dictionary. 
Each state in the manifest corresponds to a state in the state diagram.


In [2]:
STATE_MANIFEST_V1 = {
        "INIT": {
            "input_data": {
                "name": "Sara",
                "user_message": "Write a one sentence story"
            }
        },
        "GENERATE": {
            "module_path": "gai.asm.states",
            "class_name": "PureActionState",
            "title": "GENERATE",
            "action": "generate",
            "input_data": {
                "llm_config": {
                    "type": "getter",
                    "dependency": "get_llm_config"
                },
            },
            "output_data": [
                "streamer",
                "get_assistant_message"
            ]
        },
        "FINAL": {
            "output_data": ["monologue_messages"]
        }
    }


### c) Create state action

In [3]:
from gai.asm import AsyncStateMachine
from gai.chat.openai import AsyncOpenAI

async def generate_action(state):
    
    llm_config = state.machine.state_bag["llm_config"]
    client = AsyncOpenAI(llm_config)
    
    # Import data from state_bag
    user_message = state.machine.state_bag.get("user_message", "If you are seeing this, that means I have forgotten to add a user message. Remind me.")
    
    # Execute
    
    response = await client.chat.completions.create(
        model=llm_config["model"],
        messages=[{
            "role":"user",
            "content":user_message
            }],
        max_tokens=50,
        stream=True
    )
    
    assistant_message = ""
    async def streamer():
        nonlocal assistant_message
        async for chunk in response:
            chunk = chunk.choices[0].delta.content
            if isinstance(chunk,str) and chunk:
                assistant_message += chunk
                yield chunk

    state.machine.state_bag["get_assistant_message"] = lambda: assistant_message
    state.machine.state_bag["streamer"] = streamer()

### c) Build State Machine

In [4]:
from gai.asm import AsyncStateMachine

with AsyncStateMachine.StateMachineBuilder(STATE_DIAGRAM) as builder:
    fsm = builder.build(
        STATE_MANIFEST_V1,
        get_llm_config=lambda state: {
            "client_type": "gai",
            "name": "dolphin_llama:exl2",
            "model": "ttt",
            "url": "http://gai-chat-svr:12031/gen/v1/chat/completions"
        },
        generate=generate_action
        )


### d) Run State Machine (INIT->GENERATE)

In [5]:
await fsm.run_async()
async for chunk in fsm.state_bag["streamer"]:
    print(chunk,end='',flush=True)
print("\n\n")

In the heart of a bustling city, amidst towering skyscrapers and honking cars, a small, forgotten park stood as a tranquil oasis, where a lone tree quietly whispered secrets to the wind.




### d) Continue (GENERATE->FINAL)

In [ ]:
await fsm.run_async()
print("State History:")
for state in fsm.state_history:
    print(f"State: {state['state']}")
    print(f"- input: {state['input']}")
    print(f"- output: {state['output']}")
    print("-" * 20)
print("Assistant Message:")
fsm.state_bag["get_assistant_message"]()

State History:
State: INIT
- input: {'name': 'Sara', 'user_message': 'Write a one sentence story'}
- output: {'name': 'Sara', 'monologue_messages': [], 'step': 0, 'time': datetime.datetime(2025, 6, 11, 7, 7, 47, 909368)}
--------------------
State: GENERATE
- input: {'name': 'Sara', 'monologue_messages': [], 'step': 1, 'time': datetime.datetime(2025, 6, 11, 7, 7, 47, 909687), 'llm_config': {'client_type': 'gai', 'name': 'dolphin_llama:exl2', 'model': 'ttt', 'url': 'http://gai-chat-svr:12031/gen/v1/chat/completions'}}
- output: {'streamer': <async_generator object generate_action.<locals>.streamer at 0x780ff945efc0>, 'get_assistant_message': <function generate_action.<locals>.<lambda> at 0x780ff94a5a20>, 'name': 'Sara', 'monologue_messages': [], 'step': 0, 'time': datetime.datetime(2025, 6, 11, 7, 7, 48, 561276)}
--------------------
State: FINAL
- input: {'streamer': <async_generator object generate_action.<locals>.streamer at 0x780ff945efc0>, 'get_assistant_message': <function generat